In [ ]:
from scripts.data_fetcher import StockDataFetcher
from scripts.feature_engineer import StockFeatureEngineer

In [ ]:
# Initialize the fetcher
fetcher = StockDataFetcher()

# Fetch and store NASDAQ data
fetcher.fetch_and_store_nasdaq_data(years=1)


# Or fetch data for specific tickers
# specific_data = fetcher.load_data(['AAPL', 'GOOGL', 'MSFT'], years=5)
# processed_data = fetcher.process_price_data(specific_data)
# fetcher.save_to_hdf5(processed_data, price_key='custom/price')

INFO:data_fetcher:Retrieved 3825 NASDAQ tickers
INFO:data_fetcher:Downloading data from 2024-05-21 to 2025-05-21
[*                      2%                       ]  77 of 3825 completed

KeyboardInterrupt: 

[*                      3%                       ]  108 of 3825 completed

In [ ]:
feature_engineer = StockFeatureEngineer(
    data_store_path='./data/nasdaq_data_yf.h5',
    start_date='2009-01-01',
    end_date='2025-04-25',
    min_years=7,
    top_n_stocks=1000
)

# Run the complete pipeline
processed_data = feature_engineer.run_full_pipeline(save_output=True,output_path='./data/data_yf.h5')

# Get summary statistics
print("\nreturn features summary:")
print(feature_engineer.get_feature_summary(processed_data))


INFO:feature_engineer:Starting feature engineering pipeline...
INFO:feature_engineer:Loaded data: 7918243 price records, 3818 metadata records
INFO:feature_engineer:Filtered to 1847 stocks with ≥7 years of data
INFO:feature_engineer:Aligned data: 1845 common symbols
INFO:feature_engineer:Selected universe: top 1000 stocks by market cap
INFO:feature_engineer:Computed dollar volume features
INFO:feature_engineer:Computed RSI
INFO:feature_engineer:Computed Bollinger Bands
INFO:feature_engineer:Computed ATR features
INFO:feature_engineer:Computed MACD features
INFO:feature_engineer:Added sector encoding
INFO:feature_engineer:Computed returns for periods: [1, 5, 10, 21, 42, 63]
INFO:feature_engineer:Computed return deciles
INFO:feature_engineer:Computed sector return quintiles
INFO:feature_engineer:Computed forward returns
INFO:feature_engineer:Removed 73 outlier stocks with daily returns > 100.0%
INFO:feature_engineer:Added time features
INFO:feature_engineer:Saved processed data to data_y


Return Features Summary:
                r01           r05  ...           r42           r63
count  3.365643e+06  3.361935e+06  ...  3.327636e+06  3.308169e+06
mean   7.852169e-04  3.787612e-03  ...  3.128987e-02  4.743401e-02
std    2.890796e-02  6.333625e-02  ...  1.879987e-01  2.382892e-01
min   -8.453237e-01 -8.503937e-01  ... -9.070728e-01 -9.024919e-01
25%   -1.104765e-02 -2.444798e-02  ... -6.219846e-02 -7.119184e-02
50%    0.000000e+00  2.153606e-03  ...  1.998933e-02  3.031789e-02
75%    1.195218e-02  2.967278e-02  ...  1.079822e-01  1.403614e-01
max    1.000000e+00  2.454839e+00  ...  8.479167e+00  1.857286e+01

[8 rows x 6 columns]


In [ ]:
# to plot distributions
# feature_engineer.plot_rsi_distribution(processed_data)
# feature_engineer.plot_bollinger_bands_distribution(processed_data)

In [7]:
import pandas as pd
START = '2009-01-01'
END = '2020-01-01'
idx = pd.IndexSlice
DATA_STORE = './data/nasdaq_data_yf_test.h5'
ohlcv = ['open', 'close', 'low', 'high', 'volume']
with pd.HDFStore(DATA_STORE) as store:
    prices = (store['nasdaq/price']
            .loc[idx[START:END, :], ohlcv] # select OHLCV columns from 2010 until 2017
            #.rename(columns=lambda x: x.replace('adj_', '')) # simplify column names
            .swaplevel()
            .sort_index())
    metadata = (store['nasdaq/metadata'].loc[:, ['marketcap', 'sector']])

In [8]:
prices

Price                  open     close       low      high   volume
ticker date                                                       
AACG   2010-01-04  4.650000  0.377391  4.650000  4.750000   1100.0
       2010-01-05  4.740000  0.373376  4.650000  4.740000    300.0
       2010-01-06  4.510000  0.362135  4.510000  4.510000    100.0
       2010-01-07  4.520000  0.362938  4.520000  4.520000    200.0
       2010-01-08  4.740000  0.381406  4.530000  4.750000   1700.0
...                     ...       ...       ...       ...      ...
ZYXI   2010-12-27  0.527273  0.269047  0.500000  0.563636  28820.0
       2010-12-28  0.518182  0.269047  0.454545  0.518182  23980.0
       2010-12-29  0.509091  0.283723  0.509091  0.527273   9900.0
       2010-12-30  0.527273  0.303289  0.527273  0.563636   4290.0
       2010-12-31  0.527273  0.293506  0.527273  0.563636  43450.0

[306933 rows x 5 columns]